# 04 — Evaluation

Metrics, confusion matrices, calibration and threshold analysis for both models, evaluated on the leak-free test split (`03_Training.ipynb`). Every number below comes from a real `results/*.json` file, regenerated by `backend/scripts/regenerate_metrics.py` -- never hand-edited (`backend/scripts/verify_metrics_freshness.py` is the CI guard against that drift).

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
import json

metrics = json.load(open("results/reproduced_metrics.json", encoding="utf-8"))
for model in ["bert", "cnn2d"]:
    t = metrics[model]["test"]["metrics"]
    print(f"{model.upper():6s}  accuracy={t['accuracy']:.4f}  f1_macro={t['f1_macro']:.4f}  "
          f"roc_auc={t['roc_auc']:.4f}   (n_test={t['n_samples']})")


**A confirmed, honestly-reported non-fix**: BERT had a blind spot on blunt late-delivery complaints. A fix attempt's *first* reported improvement turned out to be measured on data the model had just trained on; re-measured correctly on the held-out test split only, the change was not statistically distinguishable (n=94). Documented as a non-fix, not silently dropped -- see `MODEL_COMPARISON_AUDIT.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, model in zip(axes, ["bert", "cnn2d"]):
    cm = json.load(open(f"results/confusion_matrix_{model}.json", encoding="utf-8"))
    matrix = np.array(cm["matrix"])
    ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(cm["labels"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(cm["labels"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(model.upper())
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center",
                     color="white" if matrix[i, j] > matrix.max() / 2 else "black")
plt.tight_layout()
plt.show()


In [ ]:
calib = json.load(open("results/calibration_analysis.json", encoding="utf-8"))

fig, ax = plt.subplots(figsize=(7, 4))
for model, color in [("bert", "#1f77b4"), ("cnn2d", "#ff7f0e")]:
    sweep = calib[model]["threshold_sweep"]
    ax.plot([p["threshold"] for p in sweep], [p["f1"] for p in sweep],
            marker="o", label=f"{model.upper()} F1", color=color)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="Deployed threshold (0.5)")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("F1 score")
ax.set_title("Threshold sweep on the validation split")
ax.legend()
plt.tight_layout()
plt.show()


## Fake-review detector — paraphrase-stability validation

A WordNet paraphrase was generated for every one of the 320 held-out test reviews, with Wilson 95% confidence intervals computed on the resulting flip rate -- see `MODEL_COMPARISON_AUDIT.md` §9.

In [ ]:
d = json.load(open("results/fake_review_stability_largescale_test.json", encoding="utf-8"))
model_labels = {"distilbert_consistency_v4": "DistilBERT alone", "tfidf_logreg": "TF-IDF alone", "ensemble": "Ensemble"}

labels, rates, err_low, err_high = [], [], [], []
for key, label in model_labels.items():
    m = d["models"][key]
    rate = m["confident_flip_rate"]
    lo, hi = m["confident_flip_95ci"]
    labels.append(label)
    rates.append(rate * 100)
    err_low.append(max(0.0, (rate - lo) * 100))
    err_high.append(max(0.0, (hi - rate) * 100))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, rates, yerr=[err_low, err_high], capsize=6, color=["#8c564b", "#e377c2", "#2ca02c"])
ax.set_ylabel("Confident flip rate (%)")
ax.set_title(f"Paraphrase-stability: confident flip rate, 95% CI (n={d['n_test_reviews']} reviews)")
plt.tight_layout()
plt.show()
